# Notebook 08 – Backend Implementation with FastAPI

## 1. Overview

This notebook implements the backend API for the **Advance AI Research Assistant** using FastAPI.

The backend connects the previously developed PDF ingestion, text preprocessing, embedding generation, hybrid retrieval, reranking, and LLM-based answer generation components into a reusable application interface.

It provides REST API endpoints that will later be connected to the Next.js frontend.

## 2. Objectives

- Create a modular FastAPI backend.
- Organize the existing RAG components into reusable Python modules.
- Implement PDF upload and processing.
- Integrate ChromaDB for persistent vector storage.
- Connect semantic search, BM25 retrieval, and Cross-Encoder reranking.
- Integrate Ollama for local LLM-based answer generation.
- Implement API endpoints for document upload, question answering, and health checks.
- Return answers with references to the original PDF documents and page numbers.

## 3. Tools and Technologies

| Technology | Purpose |
|---|---|
| Python 3.11 | Backend programming |
| FastAPI | REST API framework |
| Uvicorn | ASGI server |
| Pydantic | Data validation and response models |
| PyMuPDF | PDF text extraction |
| Sentence Transformers | Text embeddings |
| ChromaDB | Persistent vector database |
| BM25 | Keyword-based retrieval |
| Cross-Encoder | Passage reranking |
| Ollama | Local LLM inference |
| Qwen 2.5 (3B) | Answer generation |

## 4. Input and Expected Output

**Input:**
- PDF documents uploaded through the API.
- Research questions submitted by the user.

**Expected output:**
- Extracted and indexed PDF content.
- Retrieved and reranked relevant passages.
- AI-generated answers based on retrieved evidence.
- Source references containing document filenames and page numbers.
- Appropriate error messages and responses when documents or evidence are unavailable.

## 5. Implementation Workflow

1. Inspect the existing project structure.
2. Create the required backend directory structure.
3. Organize reusable components into Python modules.
4. Implement PDF upload and document indexing.
5. Implement the RAG query pipeline.
6. Create FastAPI endpoints.
7. Start the API using Uvicorn.
8. Test the endpoints and verify the complete backend workflow.

## 6. Expected Outcome

At the end of this notebook, the **Advance AI Research Assistant** will have a functional FastAPI backend.

Users will be able to upload scientific PDF documents, submit research questions, and receive AI-generated answers supported by references to the original documents.

The backend will provide the foundation for the Next.js frontend integration.

## Step 1. -Inspect the project structure

In [1]:

from pathlib import Path

# Define the project root
PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

# Verify the project directory
assert PROJECT_ROOT.exists(), f"Project directory not found: {PROJECT_ROOT}"

print("Project root:", PROJECT_ROOT)


# Display the existing project structure
def show_project_tree(directory, max_depth=3, current_depth=0):
    """Display project folders and files up to a specified depth."""

    if current_depth > max_depth:
        return

    for path in sorted(directory.iterdir()):
        # Skip large or unnecessary directories
        if path.name in {
            ".git", ".venv", "__pycache__",
            "node_modules", ".next", ".ipynb_checkpoints"
        }:
            continue

        indent = "    " * current_depth

        if path.is_dir():
            print(f"{indent}[DIR]  {path.name}/")
            if current_depth < max_depth:
                show_project_tree(
                    path,
                    max_depth=max_depth,
                    current_depth=current_depth + 1
                )
        else:
            print(f"{indent}[FILE] {path.name}")


show_project_tree(PROJECT_ROOT, max_depth=3)


Project root: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant
[FILE] .env.example
[FILE] .gitignore
[FILE] SETUP-GUIDE.md
[DIR]  backend/
    [FILE] __init__.py
    [DIR]  app/
        [FILE] __init__.py
    [DIR]  tests/
        [FILE] __init__.py
[DIR]  benchmarks/
[DIR]  data/
    [DIR]  chroma_db/
        [DIR]  68e69dee-38f8-47e3-a4bb-cc43d8c052ea/
            [FILE] data_level0.bin
            [FILE] header.bin
            [FILE] length.bin
            [FILE] link_lists.bin
        [FILE] chroma.sqlite3
    [DIR]  processed/
        [FILE] document_chunks.jsonl
        [FILE] retrieval_test_results.csv
        [FILE] semantic_search_test.csv
    [DIR]  raw_pdfs/
        [FILE] 41598_2019_Article_38798.pdf
        [FILE] NIPS-2017-attention-is-all-you-need-Paper.pdf
        [FILE] s41598-019-52937-5.pdf
    [DIR]  sample_documents/
    [DIR]  uploads/
[DIR]  docs/
[FILE] environment.yml
[DIR]  experiments/
[DIR]  frontend/
[DIR]  logs/
[DIR]  notebooks/


## Step 2 – Inspect the existing backend files

In [2]:

# Inspect existing backend and ingestion files

folders_to_inspect = [
    PROJECT_ROOT / "backend",
    PROJECT_ROOT / "src",
]

for folder in folders_to_inspect:
    print(f"\n{'=' * 65}")
    print(f"FOLDER: {folder.relative_to(PROJECT_ROOT)}")
    print("=" * 65)

    for path in sorted(folder.rglob("*")):
        if any(
            part in {".git", "__pycache__", ".ipynb_checkpoints"}
            for part in path.parts
        ):
            continue

        if path.is_file():
            print(path.relative_to(PROJECT_ROOT))


# Inspect the existing PDF extractor
extractor_file = PROJECT_ROOT / "src" / "ingestion" / "pdf_extractor.py"

if extractor_file.exists():
    print("\n" + "=" * 65)
    print("EXISTING PDF EXTRACTOR")
    print("=" * 65)

    print(extractor_file.read_text(encoding="utf-8"))
else:
    print("\nNo existing PDF extractor found.")



FOLDER: backend

FOLDER: src
src/ingestion/__init__.py
src/ingestion/pdf_extractor.py

EXISTING PDF EXTRACTOR

from pathlib import Path
import fitz


def extract_pdf(pdf_path: str | Path) -> dict:
    """
    Validate a PDF and extract its metadata and page-level text.

    Args:
        pdf_path: Path to the PDF document.

    Returns:
        Dictionary containing document metadata and extracted pages.
    """

    pdf_path = Path(pdf_path)

    # Validate the input file.
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError("The file must have a PDF extension.")

    if pdf_path.stat().st_size == 0:
        raise ValueError("The PDF file is empty.")

    pages = []

    try:
        with fitz.open(pdf_path) as document:

            if document.is_encrypted:
                raise ValueError("Password-protected PDFs are not supported.")

            if document.page_count == 0:


## Step 3 – Create the backend package structure

In [3]:

# Create the backend package structure

directories = [
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "src" / "processing",
    PROJECT_ROOT / "src" / "embeddings",
    PROJECT_ROOT / "src" / "retrieval",
    PROJECT_ROOT / "src" / "generation",
    PROJECT_ROOT / "backend" / "app",
    PROJECT_ROOT / "backend" / "tests",
]

# Create directories if they do not already exist
for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)


# Create package initialization files
init_files = [
    PROJECT_ROOT / "src" / "__init__.py",
    PROJECT_ROOT / "src" / "processing" / "__init__.py",
    PROJECT_ROOT / "src" / "embeddings" / "__init__.py",
    PROJECT_ROOT / "src" / "retrieval" / "__init__.py",
    PROJECT_ROOT / "src" / "generation" / "__init__.py",
    PROJECT_ROOT / "backend" / "__init__.py",
    PROJECT_ROOT / "backend" / "app" / "__init__.py",
    PROJECT_ROOT / "backend" / "tests" / "__init__.py",
]

for file_path in init_files:
    file_path.touch(exist_ok=True)

print("Backend package structure created successfully.")

# Verify the new structure
for directory in directories:
    print(f"✓ {directory.relative_to(PROJECT_ROOT)}")


Backend package structure created successfully.
✓ src
✓ src/processing
✓ src/embeddings
✓ src/retrieval
✓ src/generation
✓ backend/app
✓ backend/tests


## Step 4 – Verify the result

In [4]:
show_project_tree(PROJECT_ROOT, max_depth=3)

[FILE] .env.example
[FILE] .gitignore
[FILE] SETUP-GUIDE.md
[DIR]  backend/
    [FILE] __init__.py
    [DIR]  app/
        [FILE] __init__.py
    [DIR]  tests/
        [FILE] __init__.py
[DIR]  benchmarks/
[DIR]  data/
    [DIR]  chroma_db/
        [DIR]  68e69dee-38f8-47e3-a4bb-cc43d8c052ea/
            [FILE] data_level0.bin
            [FILE] header.bin
            [FILE] length.bin
            [FILE] link_lists.bin
        [FILE] chroma.sqlite3
    [DIR]  processed/
        [FILE] document_chunks.jsonl
        [FILE] retrieval_test_results.csv
        [FILE] semantic_search_test.csv
    [DIR]  raw_pdfs/
        [FILE] 41598_2019_Article_38798.pdf
        [FILE] NIPS-2017-attention-is-all-you-need-Paper.pdf
        [FILE] s41598-019-52937-5.pdf
    [DIR]  sample_documents/
    [DIR]  uploads/
[DIR]  docs/
[FILE] environment.yml
[DIR]  experiments/
[DIR]  frontend/
[DIR]  logs/
[DIR]  notebooks/
    [FILE] 00_project_setup.ipynb
    [FILE] 01_environment_and_dependencies.ipynb
    [F

## Step 5 – Implement the text processing module
Now we will move the preprocessing and chunking logic from Notebook 04 into a reusable Python module.

File to create:

src/processing/text_processor.py

**This module will:**
1. Normalize extracted PDF text.
2. Remove unnecessary whitespace and line-break hyphenation.
3. Split each page into overlapping chunks.
4. Preserve document ID, filename, page number, and chunk index.
5. Generate unique chunk IDs.
   
**We will keep the same settings used in Notebook 04:**
- Chunk size: 1200 characters
- Chunk overlap: 150 characters
- Page-wise chunking: enabled

**5.1 Create the Python module**

In [5]:

# Create the text processing module

from pathlib import Path

processor_file = (
    PROJECT_ROOT
    / "src"
    / "processing"
    / "text_processor.py"
)

processor_code = r'''
import re
import unicodedata
from pathlib import Path


# Default chunking configuration
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 150


def clean_text(text: str) -> str:
    """
    Normalize extracted scientific PDF text.

    Preserves original scientific terminology and symbols.
    """

    if not isinstance(text, str):
        raise TypeError("Input text must be a string.")

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Remove soft hyphens
    text = text.replace("\u00ad", "")

    # Join words split by a hyphen at a line break
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # Normalize line breaks and whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def create_chunks(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP
) -> list:
    """
    Split text into overlapping character-based chunks.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")

    if chunk_overlap < 0 or chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must be non-negative and smaller than chunk_size."
        )

    text = clean_text(text)

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):

        end = min(start + chunk_size, len(text))

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == len(text):
            break

        start = end - chunk_overlap

    return chunks


def process_document(
    extracted_document: dict,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP
) -> list:
    """
    Convert an extracted PDF document into page-level chunks.

    Each chunk preserves its original document and page information.
    """

    filename = extracted_document["filename"]

    document_id = Path(filename).stem

    processed_chunks = []

    for page in extracted_document["pages"]:

        page_number = page["page_number"]

        page_chunks = create_chunks(
            page["text"],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        for chunk_index, chunk_text in enumerate(page_chunks, start=1):

            chunk_id = (
                f"{document_id}_p{page_number}_c{chunk_index}"
            )

            processed_chunks.append({
                "chunk_id": chunk_id,
                "document_id": document_id,
                "filename": filename,
                "page_number": page_number,
                "chunk_index": chunk_index,
                "text": chunk_text,
                "character_count": len(chunk_text),
                "word_count": len(chunk_text.split())
            })

    return processed_chunks
'''

processor_file.write_text(processor_code.strip() + "\n", encoding="utf-8")

print("Created:", processor_file)


Created: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/src/processing/text_processor.py


**5.2 Test the module**

In [3]:

# Import the new processing functions

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.processing.text_processor import (
    clean_text,
    create_chunks,
    process_document
)

# Test text cleaning
sample_text = "Deep learn-\ning models   improve scientific analysis."

cleaned = clean_text(sample_text)

print("Cleaned text:")
print(cleaned)


# Test chunk creation
sample_chunks = create_chunks(
    "This is a sample scientific document. " * 100
)

print("\nNumber of test chunks:", len(sample_chunks))
print("First chunk length:", len(sample_chunks[0]))


# Test document processing using an existing PDF
from src.ingestion.pdf_extractor import extract_pdf

sample_pdf = (
    PROJECT_ROOT
    / "data"
    / "raw_pdfs"
    / "41598_2019_Article_38798.pdf"
)

extracted_document = extract_pdf(sample_pdf)

processed_chunks = process_document(extracted_document)

print("\nDocument processing results")
print("Filename:", extracted_document["filename"])
print("Pages:", extracted_document["page_count"])
print("Generated chunks:", len(processed_chunks))

print("\nFirst chunk:")
print(processed_chunks[0])


Cleaned text:
Deep learning models improve scientific analysis.

Number of test chunks: 4
First chunk length: 1200

Document processing results
Filename: 41598_2019_Article_38798.pdf
Pages: 14
Generated chunks: 60

First chunk:
{'chunk_id': '41598_2019_Article_38798_p1_c1', 'document_id': '41598_2019_Article_38798', 'filename': '41598_2019_Article_38798.pdf', 'page_number': 1, 'chunk_index': 1, 'text': 'www.nature.com/scientificreports OPEN Cell Identity Codes: Understanding Cell Identity from Gene Expression Profiles using Deep Neural Received: 5 September 2017Accepted: 10 January 2019 Networks Published: xx xx xxxx Farzad Abdolhosseini1, Behrooz Azarkhalili2, Abbas Maazallahi1, Aryan Kamal1, Seyed Abolfazl Motahari1, Ali Sharifi-Zarchi 1 & Hamidreza Chitsaz3 Understanding cell identity is an important task in many biomedical areas. Expression patterns of specific marker genes have been used to characterize some limited cell types, but exclusive markers are not available for many cell

In [4]:

import json

chunks_file = PROJECT_ROOT / "data" / "processed" / "document_chunks.jsonl"

with chunks_file.open("r", encoding="utf-8") as file:
    existing_chunks = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

print("Existing dataset:", len(existing_chunks), "chunks")

print("\nFirst three existing chunk IDs:")
for chunk in existing_chunks[:3]:
    print(chunk["chunk_id"])

print("\nNew processor chunk IDs:")
for chunk in processed_chunks[:3]:
    print(chunk["chunk_id"])


Existing dataset: 156 chunks

First three existing chunk IDs:
41598_2019_Article_38798_p1_c1
41598_2019_Article_38798_p1_c2
41598_2019_Article_38798_p1_c3

New processor chunk IDs:
41598_2019_Article_38798_p1_c1
41598_2019_Article_38798_p1_c2
41598_2019_Article_38798_p1_c3
